# Demo: Offline Indexing + Online Retrieval (Visual Product Search)

Notebook nay minh hoa end-to-end 2 chu trinh trong README.md:
- **Hinh 1.1**: build Static Main HNSW tu catalog, roi tim kiem baseline.
- **Hinh 2.9**: them/go san pham (Dynamic Buffer + Blacklist), tim kiem cai tien.

Chay notebook nay tu thu muc goc repo (de import duoc `common`, `offline`, `online`).

In [ ]:
import sys
sys.path.append('..')  # neu chay notebook tu ben trong notebooks/

from common.config import load_config
from common.backbone.dinov3_encoder import DINOv3Encoder

## 1. Offline: build Static Main HNSW (Hinh 1.1, box tren)

In [ ]:
from offline.feature_extraction.sop_dataset import SOPDataset
from offline.feature_extraction.extract_embeddings import extract_catalog_embeddings
from offline.indexing.build_index import build_hnsw_index, save_index

offline_cfg = load_config('../configs/offline.yaml')

dataset = SOPDataset(root=offline_cfg['dataset']['root'], list_file=offline_cfg['dataset']['catalog_list'])
encoder = DINOv3Encoder.from_config(offline_cfg['model'])

embeddings, product_ids = extract_catalog_embeddings(dataset, encoder, batch_size=offline_cfg['model']['batch_size'])
index = build_hnsw_index(embeddings, m=offline_cfg['index']['hnsw_m'], ef_construction=offline_cfg['index']['ef_construction'])
save_index(index, product_ids, offline_cfg['paths']['static_index_path'], offline_cfg['paths']['static_id_map_path'])
print(f"Static HNSW: {index.ntotal} san pham")

## 2. Online: tim kiem baseline (Hinh 1.1, box duoi)

In [ ]:
from common.preprocessing.image_io import load_image
from online.retrieval.ann_search import ANNSearcher
from online.reranking.text_rerank import top_k

online_cfg = load_config('../configs/online.yaml')
searcher = ANNSearcher(online_cfg['paths']['static_index_path'], online_cfg['paths']['static_id_map_path'], ef_search=online_cfg['retrieval']['ef_search'])

query_image = load_image('PATH/TO/QUERY.jpg')
query_vec = encoder.encode([query_image])
scores, ids = searcher.search(query_vec, top_n=online_cfg['retrieval']['top_n_candidates'])
scores, ids = top_k(scores[0], ids[0], online_cfg['rerank']['top_k'])
list(zip(ids, scores))

## 3. Online cai tien: Dynamic Buffer + Blacklist + Text Re-ranking (Hinh 2.9)

Xem `online/pipelines/run_online_retrieval_dynamic.py` va `online/pipelines/manage_dynamic_catalog.py`
cho ban CLI day du (search song song qua `ThreadPoolExecutor`, gop ung vien, loc blacklist, text rerank).